# TechForge Step 6 - DevDemo notebook

Runs the whole prototype of Step 4 end to end (Table 12 steps 1-11), runs the tests, and saves the
report figures (PNG) and result tables (CSV) to `report_outputs/`.

* **Real data:** pc-part-dataset (commit c52a04c) - product specifications for rules R1-R4.
* **SYNTHETIC data:** builder events, stock, sales, true demand, lead times - every result based on them is labelled "synthetic".
* **Not shown here:** rule accuracy and Cohen's kappa - they need the team's labels (`scripts/evaluate_labels.py`).

In [ ]:
import sys, time, subprocess
from pathlib import Path
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option("display.max_colwidth", 140); pd.set_option("display.width", 200)
import run_pipeline
from techforge import lake, report_figures
from techforge.step02_generate import GeneratorSettings

RUN_ID = "demo"          # re-running replaces this run's outputs
ENGINE = "pandas"        # "spark" needs pyspark + Java (optional)
PART_KEY = "extended"    # D1 (P8): name + variant fields; "report" = original key, for comparison

## A. Pipeline steps 1-11 (Table 12)

In [ ]:
t0 = time.perf_counter()
results = run_pipeline.run(RUN_ID, list(range(1, 12)), engine=ENGINE, part_key=PART_KEY,
                           settings=GeneratorSettings(seed=42))
print(f"total {time.perf_counter() - t0:.1f} s")

### Step 3-4: part merging and UNKNOWN share per attribute (real catalog)

In [ ]:
display(lake.read_table("cleaned", "parts_cleaning_log", RUN_ID))
display(lake.read_table("curated", "attribute_coverage", RUN_ID))

### Step 5: quarantined and late events (SYNTHETIC)

In [ ]:
display(lake.read_table("cleaned", "event_validation_log", RUN_ID))

### Step 7-8: model and replay (SYNTHETIC events, real catalog)

In [ ]:
print(results[7])
print(results[8])
display(lake.read_table("outputs", "reco_next_part", RUN_ID).head(10))

### Step 9: Focus A slice (SYNTHETIC)

In [ ]:
print(results[9])
display(lake.read_table("outputs", "fa_alert", RUN_ID).head())

## B. Table 11 - evaluation results
Accuracy rows remain *pending* until the team's labels exist. All other rows are mechanics checks on SYNTHETIC data.

In [ ]:
display(lake.read_table("outputs", "evaluation_results", RUN_ID)[["test", "measure", "value", "target", "status", "data", "note"]])

In [ ]:
display(lake.read_table("outputs", "eval_injected_detection", RUN_ID))
display(lake.read_table("outputs", "eval_forecast_metrics", RUN_ID))
display(lake.read_table("outputs", "eval_sim_summary", RUN_ID))

## C. Tests (pytest)

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-2000:])

## D. Report figures and tables -> `report_outputs/`

In [ ]:
paths = report_figures.write_all(RUN_ID)
for p in paths: print(p.relative_to(ROOT))

In [ ]:
from IPython.display import Image
for p in [p for p in paths if p.suffix == ".png"]:
    display(Image(filename=str(p)))

## E. Label sample for the team (no labels are created here)

In [ ]:
r = subprocess.run([sys.executable, "scripts/draw_label_sample.py", "--run-id", RUN_ID], cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-2500:]); print(r.stderr[-1500:])

## F. Data lineage: catalog of this run

In [ ]:
cat = lake.read_catalog(RUN_ID)
pd.DataFrame(cat["entries"].values())[["zone", "dataset", "row_count", "source", "code_version", "reference_version", "created_at"]]